# SQL Query Runtime Predictor 🚀

This project predicts the execution time of SQL queries on a MySQL database using classical machine learning models trained on features extracted from the query text and the database's `EXPLAIN` execution plans.

## Project Workflow
1. **Database Setup & Seeding**: Generate 10k customers, 5k products, and 50k orders with Faker.
2. **Query Generation**: Programmatically generate 5,000+ SQL queries ranging in complexity (JOINs, filters, GROUP BY, ORDER BY).
3. **Data Collection**: Measure query runtimes (ms) and extract optimizer features via `EXPLAIN FORMAT=TRADITIONAL`.
4. **Feature Engineering**: Standard scale numerical features and apply log-transforms to skewed fields.
5. **Machine Learning**: Train and compare Linear Regression, Random Forest, and XGBoost regressors using 5-fold cross-validation.
6. **Query Optimization Advisor**: Implement a recommendation system for slow queries.

### 1. Initialization and Imports

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Add src to path
sys.path.append(os.path.abspath('../'))
from src.analysis import recommend_optimizations

print("Libraries loaded successfully.")

### 2. Dataset Overview
Let's load the collected dataset of 5,000 queries and display the first few rows.

In [ ]:
df = pd.read_csv('../query_dataset.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

### 3. Feature Correlations & Distributions
Let's look at the correlation between query features and execution time. We can inspect the generated plots.

In [ ]:
# Display correlation heatmap
display(Image(filename='../plots/correlation_matrix.png'))

Let's check the execution time distribution on a normal scale and a log scale.

In [ ]:
display(Image(filename='../plots/target_distribution.png'))

### 4. Database Performance Analysis

#### Impact of JOINs
Let's visualize the runtime distribution grouped by the number of joins.

In [ ]:
display(Image(filename='../plots/query_performance_by_joins.png'))

Let's display the mean, median, and count statistics for joins and index usage.

In [ ]:
join_stats = df.groupby('num_joins')['execution_time'].agg(['count', 'mean', 'median', 'std']).reset_index()
print("=== JOIN Impact Statistics ===")
print(join_stats.to_string(index=False))

print("\n=== Index Impact Statistics (Single-Table Filtered Queries) ===")
single_table_filtered = df[(df['num_filters'] > 0) & (df['num_joins'] == 0)]
index_stats = single_table_filtered.groupby('uses_index')['execution_time'].agg(['count', 'mean', 'median', 'std']).reset_index()
print(index_stats.to_string(index=False))

### 5. Model Evaluation and Comparison
Let's see how our trained models compare on the test set.

In [ ]:
display(Image(filename='../plots/model_comparison.png'))

### 6. Model Feature Importance & Error Distribution
Let's check which features the best model relies on to predict the runtimes, and visualize the error distributions.

In [ ]:
display(Image(filename='../plots/feature_importance.png'))

In [ ]:
display(Image(filename='../plots/actual_vs_predicted.png'))

In [ ]:
display(Image(filename='../plots/error_distribution.png'))

### 7. Interactive Query Optimization Advisor (Bonus)
Use our recommendation function to predict the execution time of any custom SQL query and get immediate suggestions for indexing, query restructuring, and optimization.

In [ ]:
# Example 1: Slow Query (Multiple Joins + Non-indexed filter + Order By)
query_1 = """
SELECT * FROM Orders O 
JOIN Customers C ON O.customer_id = C.id 
JOIN Products P ON O.product_id = P.id 
WHERE C.state = 'California' AND P.rating < 2.0
ORDER BY O.total_amount DESC
"""

result_1 = recommend_optimizations(query_1)
print(f"Query:\n{query_1.strip()}")
print(f"\nPredicted Runtime: {result_1['predicted_runtime_ms']} ms")
print(f"Uses Index: {result_1['uses_index']}")
print(f"Estimated Rows Search Space: {result_1['estimated_rows']}")
print("\nAdvisor Recommendations:")
for rec in result_1['recommendations']:
    print(rec)

In [ ]:
# Example 2: Fast Query (Single-table + Primary key filter)
query_2 = "SELECT * FROM Products WHERE id = 125"
result_2 = recommend_optimizations(query_2)
print(f"Query: {query_2}")
print(f"Predicted Runtime: {result_2['predicted_runtime_ms']} ms")
print(f"Uses Index: {result_2['uses_index']}")
print(f"Estimated Rows Search Space: {result_2['estimated_rows']}")
print("\nAdvisor Recommendations:")
for rec in result_2['recommendations']:
    print(rec)